In [ ]:
import pandas as pd

In [ ]:
trans_clean = pd.read_csv("transactions_cleaned.csv")
subs = pd.read_csv("subscriptions.csv")
users = pd.read_csv("users.csv")

In [ ]:
merged = trans_clean.merge(
    users,
    on="user_id",
    how="left"
)

print(merged.to_string())

In [ ]:
merged[['acquisition_channel', 'country', 'subscription_plan']].isnull().sum()

In [ ]:
acquisition_channel_order_value = merged.groupby('acquisition_channel')['order_value'].sum()
geographic_order_value = merged.groupby('country')['order_value'].sum()
subscription_plan_order_value = merged.groupby('subscription_plan')['order_value'].sum()
devices_order_value = merged.groupby('device')['order_value'].sum()

print(acquisition_channel_order_value)
print(geographic_order_value)
print(subscription_plan_order_value)
print(devices_order_value)

In [ ]:
#calculate the number of orders for each acquisition channel, country, subscription plan, and device type
channel_orders = (
    merged.groupby('acquisition_channel')['transaction_id'].count().reset_index(name='total_orders')
)

country_orders = (
    merged.groupby('country')['transaction_id'].count().reset_index(name='total_orders')
)

plan_orders = (
    merged.groupby('subscription_plan')['transaction_id'].count().reset_index(name='total_orders')
)

device_orders = (
    merged.groupby('device')['transaction_id'].count().reset_index(name='total_orders')
)

print(channel_orders)
print(country_orders)
print(plan_orders)
print(device_orders)

In [ ]:
#calculate the average order value 

Channel_AOV = acquisition_channel_order_value/channel_orders.set_index('acquisition_channel')['total_orders'].round(2)
Country_AOV = geographic_order_value/country_orders.set_index('country')['total_orders'].round(2)
Plan_AOV = subscription_plan_order_value/plan_orders.set_index('subscription_plan')['total_orders'].round(2)
Device_AOV = devices_order_value/device_orders.set_index('device')['total_orders'].round(2)

print(Channel_AOV)
print(Country_AOV)
print(Plan_AOV)
print(Device_AOV)

In [ ]:
#calculate the number of unique customers for each acquisition channel, country, subscription plan, and device type
uniq_cust_ch = merged.groupby('acquisition_channel')['user_id'].nunique()
uniq_cust_country = merged.groupby('country')['user_id'].nunique()
uniq_cust_plan = merged.groupby('subscription_plan')['user_id'].nunique()
uniq_cust_device = merged.groupby('device')['user_id'].nunique()

channel_orders_series = channel_orders.set_index('acquisition_channel')['total_orders']
country_orders_series = country_orders.set_index('country')['total_orders']
plan_orders_series = plan_orders.set_index('subscription_plan')['total_orders']
device_orders_series = device_orders.set_index('device')['total_orders']

Purchase_frequency_ch = (channel_orders_series / uniq_cust_ch).round(2)
Purchase_frequency_country = (country_orders_series / uniq_cust_country).round(2)
Purchase_frequency_plan = (plan_orders_series / uniq_cust_plan).round(2)
Purchase_frequency_device = (device_orders_series / uniq_cust_device).round(2)

print(Purchase_frequency_ch)
print(Purchase_frequency_country)
print(Purchase_frequency_plan)
print(Purchase_frequency_device)

In [ ]:
customer_lifespan = (
    merged.groupby('user_id')
    .agg(
        first_purchase=('transaction_date', 'min'),
        last_purchase=('transaction_date', 'max')
    )
    .reset_index()
)

In [ ]:
#Convert the first_purchase and last_purchase columns to datetime format
customer_lifespan['first_purchase'] = pd.to_datetime(
    customer_lifespan['first_purchase']
)

customer_lifespan['last_purchase'] = pd.to_datetime(
    customer_lifespan['last_purchase']
)

#Calculate the customer lifespan in months
customer_lifespan['customer_lifespan_months'] = (
    (customer_lifespan['last_purchase'].dt.year - customer_lifespan['first_purchase'].dt.year) * 12
    +
    (customer_lifespan['last_purchase'].dt.month - customer_lifespan['first_purchase'].dt.month)
)

In [ ]:
#Merge the customer_lifespan DataFrame with the users 
#DataFrame to include acquisition_channel, country, subscription_plan, and device information
customer_lifespan = customer_lifespan.merge(
    users[['user_id', 'acquisition_channel', 'country', 'subscription_plan', 'device']],
    on='user_id',
    how='left'
)

In [ ]:
#calculate the average customer lifespan for each acquisition channel, country, subscription plan, and device type
channel_lifespan = (
    customer_lifespan
    .groupby('acquisition_channel')['customer_lifespan_months']
    .mean()
    .reset_index(name='avg_customer_lifespan')
)

print(channel_lifespan)

country_lifespan = (
    customer_lifespan
    .groupby('country')['customer_lifespan_months']
    .mean()
    .reset_index(name='avg_customer_lifespan')
)

print(country_lifespan)

subscription_plan_lifespan = (
    customer_lifespan
    .groupby('subscription_plan')['customer_lifespan_months']
    .mean()
    .reset_index(name='avg_customer_lifespan')
)

print(subscription_plan_lifespan)

device_lifespan = (
    customer_lifespan
    .groupby('device')['customer_lifespan_months']
    .mean()
    .reset_index(name='avg_customer_lifespan')
)

print(device_lifespan)


In [ ]:
#calculate the customer lifetime value (CLTV) for each acquisition channel, country, subscription plan, and device type
Channel_CLTV = Channel_AOV * Purchase_frequency_ch * channel_lifespan.set_index('acquisition_channel')['avg_customer_lifespan'].round(2)
Country_CLTV = Country_AOV * Purchase_frequency_country * country_lifespan.set_index('country')['avg_customer_lifespan'].round(2)
Plan_CLTV = Plan_AOV * Purchase_frequency_plan * subscription_plan_lifespan.set_index('subscription_plan')['avg_customer_lifespan'].round(2)
Device_CLTV = Device_AOV * Purchase_frequency_device * device_lifespan.set_index('device')['avg_customer_lifespan'].round(2)

print(Channel_CLTV)
print(Country_CLTV)
print(Plan_CLTV)
print(Device_CLTV)

In [ ]:
channel_segments = pd.DataFrame({
    'Acquisition_channel': Channel_CLTV.index,
    'AOV': Channel_AOV.values.round(2),
    'Purchase_frequency_Channel': Purchase_frequency_ch.values,
    'CLTV': Channel_CLTV.values.round(2)
}).sort_values(by='CLTV', ascending=False)
print(channel_segments)
channel_segments.to_csv('channel_segments.csv', index=False)

Country_segments = pd.DataFrame({
    'Country': Country_CLTV.index,
    'AOV': Country_AOV.values.round(2),
    'Purchase_frequency_Country': Purchase_frequency_country.values,
    'CLTV': Country_CLTV.values.round(2)
}).sort_values(by='CLTV', ascending=False)

print(Country_segments)
Country_segments.to_csv('country_segments.csv', index=False)

Plan_segments = pd.DataFrame({
    'Subscription_plan': Plan_CLTV.index,
    'AOV': Plan_AOV.values.round(2),
    'Purchase_frequency_Plan': Purchase_frequency_plan.values,
    'CLTV': Plan_CLTV.values.round(2)
}).sort_values(by='CLTV', ascending=False)

print(Plan_segments)
Plan_segments.to_csv('plan_segments.csv', index=False)

Device_segments = pd.DataFrame({
    'Device': Device_CLTV.index,
    'AOV': Device_AOV.values.round(2),
    'Purchase_frequency_Device': Purchase_frequency_device.values,
    'CLTV': Device_CLTV.values.round(2)
}).sort_values(by='CLTV', ascending=False)

print(Device_segments)
Device_segments.to_csv('device_segments.csv', index=False)